# M06 — Lazy, plan y cache

[← Anterior](../M05-analisis-avanzado/03-lab-acumulados.ipynb) · [Siguiente →](02-lab-explain-dag.ipynb)

En M01 viste que `filter` no pinta filas. Aquí lo miramos por dentro: el **plan** (lo que Spark *haría*) frente al **job** (lo que *hace* cuando lanzas una acción). Y el `cache`: no es magia; es “guarda el resultado de una acción para no repetir el plan”.

Ejecuta las celdas **aquí**, en este mismo fichero (clase, juntos). Va **montado**: explicación + código + lo que tienes que ver. Lo que construyes tú está en el **lab**.

Kernel: **Python (NovaShop)**.


## Arranque

La primera celda **no es Spark todavía**: busca la raíz del repo (aunque este notebook no esté en la carpeta de arriba) y deja `RAW`, `STAGING` y `CURATED` listos. La segunda pide una `SparkSession` en `local[*]` (todos los cores de esta máquina; no hay clúster).

Al ejecutar: rutas impresas y una versión `3.5.x` con master `local[*]`.


In [ ]:
import sys
from pathlib import Path

# El notebook puede estar en trabajo/; subimos hasta encontrar el repo.
_here = Path.cwd().resolve()
ROOT = next(
    p
    for p in [_here, *_here.parents]
    if (p / "labs" / "_shared" / "session.py").is_file()
)
sys.path.insert(0, str(ROOT / "labs" / "_shared"))

from paths import RAW, STAGING, CURATED  # rutas absolutas, no Path("data/raw")
from session import get_spark

print("ROOT   ", ROOT)
print("RAW    ", RAW, "existe:", RAW.is_dir())
print("STAGING", STAGING)
print("CURATED", CURATED)


In [ ]:
# getOrCreate: si ya hay sesión en este kernel, la reusa (mismo puerto 4040)
spark = get_spark('novashop-clase-m06')
print(spark.version, spark.sparkContext.master)


## Encadenar no ejecuta

Montamos 20 filas y encadenamos dos `where` y un `select`. Eso solo alarga el plan.

Al ejecutar:

- el primer `print` es el objeto (sin número);
- `count()` sí dispara un job y te da un entero;
- `explain("formatted")` imprime el mapa: busca un *Scan* / *Filter*. No hace falta traducir cada operador.

Si tienes Spark UI en el **4040**, el Job Id no debería subir con el primer `print`; sí con el `count`.


In [ ]:
from pyspark.sql import Row
from pyspark.sql.functions import col

base = spark.createDataFrame(
    [Row(x=i, canal="web" if i % 2 == 0 else "app") for i in range(20)]
)
# Tres transformaciones: todavía no hay job
planned = base.where(col("x") > 3).where(col("canal") == "web").select("x")
print("sin acción (solo el objeto):", planned)
print("con count (ahora sí):", planned.count())
planned.explain("formatted")  # mapa, no el resultado


## El cache no se llena al escribir `.cache()`

`cache()` marca el DataFrame: “la **próxima** acción, guarda el resultado en memoria”. Si no hay `count`/`show`, la pestaña Storage de Spark UI sigue vacía.

Al ejecutar: el primer `count` materializa; el segundo debería leer de ahí. Luego `unpersist()` para no dejar basura en el Codespace. En local, con 20 filas, el cronómetro a veces no se inmuta: lo que importa es Storage, no el stopwatch.


In [ ]:
warm = planned.cache()  # aún no hay nada en Storage
print("1º count (llena el cache)", warm.count())
print("2º count (debería leer cache)", warm.count())
warm.unpersist()  # suelta la memoria


**Siguiente:** [lab de explain](02-lab-explain-dag.ipynb) sobre el fact real y la UI.
